In [13]:
import os
os.environ["PYOPENGL_PLATFORM"] = "egl"
import pyrender
import cv2
import glob
import json
import time
import trimesh
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(suppress=True, precision=3)
from scipy.spatial.transform import Rotation as R
from bpc.inference.utils.camera_utils import load_camera_params
from bpc.inference.process_pose import PoseEstimator, PoseEstimatorParams, load_pose_model
from bpc.utils.data_utils import Capture, render_mask
import bpc.utils.data_utils as du
import bpc.inference.yolo_detection_filtering as ydf

from ultralytics import YOLO

In [14]:
PHASE = 2 # 1 for phase 1, 2 for phase 2 of the OpenCV BPC challenge.

# Set the paths to the scene and models directories:

if PHASE == 1:
    scene_dir = "/home/joao/source/bpc-challenge/opencv/bpc/ipd/phase-1-backup/val/000001"
    models_dir = '/home/joao/source/bpc-challenge/opencv/bpc/ipd/phase-1-backup/models_eval'
elif PHASE == 2:
    scene_dir = "//mnt/061A31701A315E3D/ipd-dataset/bpc_baseline/datasets/phase2/train_pbr/000001"
    models_dir = '/mnt/061A31701A315E3D/ipd-dataset/bpc_baseline/datasets/phase2/models_eval'

cam_ids = ["cam1", "cam2", "cam3"]

# Get image IDs from the scene directory
image_ids = []
for filename in os.listdir(os.path.join(scene_dir, "rgb_cam1")):
    if filename.endswith(".png") or filename.endswith(".jpg"):
        image_id = int(filename.split(".")[0])
        image_ids.append(image_id)
image_ids = sorted(image_ids)  # Ensure the list is sorted

obj_ids_phase1 = [0, 1, 4, 8, 10, 11, 14, 18, 19, 20]
obj_ids_phase2 = [id for id in range(0, 10)]

if PHASE == 1:
    obj_ids = obj_ids_phase1
elif PHASE == 2:
    obj_ids = obj_ids_phase2


DEPTH_IMAGE_SCALE_PX2MM = 0.1 # from depth pixel raw value to millimeters


In [ ]:
val_depth_image_dir = "/mnt/061A31701A315E3D/ipd-dataset/bpc_baseline/datasets/phase1/val/000000/depth_cam1"
train_pbr_depth_image_dir = "/mnt/061A31701A315E3D/ipd-dataset/bpc_baseline/datasets/phase1/train_pbr/000049/depth_cam1"
val_depth_image_filename = "000003.png"
train_pbr_depth_image_filename = "000006.png"
val_depth_image = cv2.imread(os.path.join(val_depth_image_dir, val_depth_image_filename), flags=cv2.IMREAD_UNCHANGED)
train_pbr_depth_image = cv2.imread(os.path.join(train_pbr_depth_image_dir, train_pbr_depth_image_filename), flags=cv2.IMREAD_UNCHANGED)

val_hillshade_img_0 = du.hillshade_depth_image(val_depth_image, azimuth=0, is_synthetic=False)
val_hillshade_img_135 = du.hillshade_depth_image(val_depth_image, azimuth=135, is_synthetic=False)

train_pbr_hillshade_img_0 = du.hillshade_depth_image(train_pbr_depth_image, azimuth=0, is_synthetic=True)
train_pbr_hillshade_img_135 = du.hillshade_depth_image(train_pbr_depth_image, azimuth=135, is_synthetic=True)

plt.figure(figsize=(20, 10))
plt.subplot(1, 2, 1)
plt.imshow(val_hillshade_img_0, cmap='gray')
plt.title("Hillshade image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(val_hillshade_img_135, cmap='gray')
plt.title("Hillshade image")
plt.axis("off")
plt.show()

plt.figure(figsize=(20, 10))
plt.subplot(1, 2, 1)
plt.imshow(train_pbr_hillshade_img_0, cmap='gray')
plt.title("Hillshade image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(train_pbr_hillshade_img_135, cmap='gray')
plt.title("Hillshade image")
plt.axis("off")
plt.show()


In [ ]:
train_pbr_depth_image_dir = "/mnt/061A31701A315E3D/ipd-dataset/bpc_baseline/datasets/phase1/train_pbr/000049/depth_cam1"


for img_id in range(0, 5):
    train_pbr_depth_image_filename = f"{img_id:06d}.png"
    train_pbr_depth_image = cv2.imread(os.path.join(train_pbr_depth_image_dir, train_pbr_depth_image_filename), flags=cv2.IMREAD_UNCHANGED)
    train_pbr_hillshade_img_0 = du.hillshade_depth_image(train_pbr_depth_image, azimuth=0, is_synthetic=True)
    train_pbr_hillshade_img_135 = du.hillshade_depth_image(train_pbr_depth_image, azimuth=135, is_synthetic=True)
    plt.figure(figsize=(20, 10))
    plt.subplot(1, 2, 1)
    plt.imshow(train_pbr_hillshade_img_0, cmap='gray')
    plt.title(f"Hillshade image - {img_id} - azimuth 0")
    plt.axis("off")
    plt.subplot(1, 2, 2)
    plt.imshow(train_pbr_hillshade_img_135, cmap='gray')
    plt.title(f"Hillshade image - {img_id} - azimuth 135")
    plt.axis("off")
    plt.show()


In [ ]:
depth_image_dir = "/mnt/061A31701A315E3D/ipd-dataset/bpc_baseline/datasets/phase1/val/000006/depth_cam1"
depth_image_filename = "000000.png"
depth_image = cv2.imread(os.path.join(depth_image_dir, depth_image_filename), flags=cv2.IMREAD_UNCHANGED)

depth_image = du.transform_depth_image(depth_image, DEPTH_IMAGE_SCALE_PX2MM, max_depth_mm=5000.0)

# Compute Sobel derivatives on depth_image:
sobelx = cv2.Sobel(depth_image, cv2.CV_64F, 1, 0, ksize=11)
sobely = cv2.Sobel(depth_image, cv2.CV_64F, 0, 1, ksize=11)
magnitude = np.sqrt(sobelx**2 + sobely**2)


magnitude_filtered = magnitude # cv2.GaussianBlur(magnitude, (11, 11), 0)

# plot the magnitude image, clipped at 500 mm range, log-scaled
# clip x range between 200 and 900 pixels in the plot
plt.figure(figsize=(15, 15))
plt.imshow(np.log1p(np.clip(magnitude_filtered, 0, 500000)), cmap='gray')
plt.colorbar()
plt.title('Sobel Magnitude (clipped at 500mm, log-scaled)')
#plt.xlim(200, 900)
plt.show()

print(f"Max Sobel magnitude: {np.max(magnitude)}")

magnitude_ravelled_values = magnitude.ravel()

# Create histogram of Sobel magnitude values
plt.figure(figsize=(15, 15))

# Plot full range histogram
plt.subplot(2, 2, 1)
plt.hist(magnitude_ravelled_values[magnitude_ravelled_values > 0], bins=100)
plt.yscale('log')
plt.title('Full Range Histogram of Sobel Magnitude Values (discarding zero values)')
plt.xlabel('Magnitude')
plt.ylabel('Frequency (log scale)')

# Plot zoomed histogram for values below 50. Discard zero magnitude values.
plt.subplot(2, 2, 2)

zoom_in_lower_range_magnitudes = magnitude_ravelled_values[(magnitude_ravelled_values > 0) & (magnitude_ravelled_values < 50)]
plt.hist(zoom_in_lower_range_magnitudes, bins=127)
plt.yscale('log')
plt.title('Zoomed Histogram of Sobel Magnitude Values (< 50, Zero Values Discarded)')
plt.xlabel('Magnitude')
plt.ylabel('Frequency')
plt.tight_layout()

deltas = np.diff(np.sort(magnitude_ravelled_values))

# Plot the distribution of deltas
plt.subplot(2, 2, 3)
plt.hist(deltas[deltas > 0], bins=100)
plt.yscale('log')  # Set y-axis to logarithmic scale
plt.title('Distribution of Deltas Between sorted magnitude values (discarding zero deltas)')
plt.xlabel('Delta')
plt.ylabel('Frequency')
plt.yscale('log')

# Zoom in to previous plot
plt.subplot(2, 2, 4)
plt.hist(deltas[(deltas > 0) & (deltas < 15)], bins=100)
plt.yscale('log')  # Set y-axis to logarithmic scale
plt.title('Distribution of Deltas Between sorted magnitude values (zoomed in, discarding zero deltas)')
plt.xlabel('Delta')
plt.ylabel('Frequency')
plt.yscale('log')
plt.show()





def display_depth_image(depth_image):
    # Display image:
    plt.figure(figsize=(15, 15))
    img_plot = plt.imshow(depth_image)
    plt.colorbar(img_plot, label='Depth value')
    plt.title('Depth Image with Color Grade Key')
    #plt.axis('off')
    plt.show()
display_depth_image(depth_image)



In [ ]:
# Just a scratchpad to test the du.compose_grey_plus_hillshade_depth_image function used 
# in prepare_data.py for preparing data prior to YOLO model training

dir = "/mnt/061A31701A315E3D/ipd-dataset/phase2-dataset-yolo11-all-objects/images/train"
# list all png files in this directory:

png_files = glob.glob(os.path.join(dir, "*.png"))
for png_file in png_files[:10]:
    img = cv2.imread(png_file, flags=cv2.IMREAD_UNCHANGED)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    grey_img = img[..., 0]
    hillshade_img_0 = img[..., 1]
    hillshade_img_135 = img[..., 2]

    # plot the three images:
    plt.figure(figsize=(15, 5))
    plt.subplot(1, 3, 1)
    plt.imshow(grey_img, cmap='gray')
    plt.title('Grey Image')
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.imshow(hillshade_img_0, cmap='gray')
    plt.title('Hillshade Image 0')
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.imshow(hillshade_img_135, cmap='gray')
    plt.title('Hillshade Image 135')
    plt.axis('off')

    plt.show()

In [ ]:
# This cell analyzes and visualizes the depth compression/decompression functions used to encode depth values into single bytes.
# It plots:
# 1. The compression function: y = floor(41.0 * ln(x+1)) which maps depth values in [0--500] mm to 0-255 range
# 2. The decompression function: x = exp(y/41.0) - 1 which recovers the original depth values
# 3. The identity line y=x for reference
# 4. The relative error between original and decompressed values
# This helps validate the information preservation of the compression scheme used for depth images.

# Plot functions
# given a = 41.0
# y = floor( a * ln(x+1) )  # <-- the depth compression function to encode depth values in millimeters, to a single byte
# x = exp(y / a) - 1        # <-- the depth decompression function to decode depth values from a single byte, to millimeters

# Plot the functions:
a = 41.0
x_coords = np.linspace(0, 500, 1000) # Original x-values for plotting
y_compressed = np.floor(a * np.log(x_coords + 1))
y_decompressed = np.exp(y_compressed / a) - 1

# Create figure and primary axis
fig, ax1 = plt.subplots(figsize=(10, 6)) # Adjust figsize as needed

# Plot functions on the primary y-axis (ax1)
ax1.plot(x_coords, y_compressed, label='Compressed Depth')
ax1.plot(x_coords, y_decompressed, label='Decompressed Depth')
ax1.plot(x_coords, x_coords, label='Identity (y=x)', linestyle=':', color='gray') # Identity line

ax1.set_xlabel("Original Depth (x) [mm]")
ax1.set_ylabel("Depth Values")
ax1.legend(loc='center left') # Legend for ax1 plots
ax1.grid(True)

# Create the secondary y-axis (ax2) for the relative error
ax2 = ax1.twinx()

# Calculate relative error: |x_coords - y_decompressed| / x_coords
# Handle division by zero at x_coords = 0 where error is 0.
# y_decompressed[0] is 0 when x_coords[0] is 0.
# So relative_error[0] will be 0 if denominator is non-zero.
denominator = np.maximum(x_coords, 1e-9) # Avoid division by zero; use small epsilon if x_coords is zero.
relative_error = np.abs(x_coords - y_decompressed) / denominator

# Plot relative error on ax2
ax2.plot(x_coords, relative_error, color='red', linestyle='--', label='Relative Error')
ax2.set_ylabel('Relative Error', color='red') # Label for the right y-axis
ax2.tick_params(axis='y', labelcolor='red') # Color for ticks on right y-axis
ax2.set_ylim(0.0, 0.2) # Set y-axis range for relative error
ax2.legend(loc='upper right') # Legend for ax2 plot

# Add a title to the entire plot
plt.title("Depth Compression, Decompression, and Relative Error Analysis")

fig.tight_layout() # Adjust plot layout to prevent labels/titles from overlapping
plt.show()


In [8]:
def extract_roi_with_padding(image, roi, color=(127, 127, 127)):
    """
    Extracts a ROI from an image, padding the out-of-bounds area with a specified color using cv2.copyMakeBorder().
    Args:
        image: The input image.
        roi: A tuple (x, y, w, h) defining the ROI rectangle.
        color: The padding color (default is gray).
    Returns:
        The extracted ROI with padding.
    """
    x, y, w, h = roi
    img_height, img_width = image.shape[:2]

    # Calculate padding values for each side
    top = max(0, -y)
    bottom = max(0, (y + h) - img_height)
    left = max(0, -x)
    right = max(0, (x + w) - img_width)

    # Calculate the ROI within the image boundaries
    x_start = max(0, x)
    y_start = max(0, y)
    x_end = min(img_width, x + w)
    y_end = min(img_height, y + h)

    # Extract the ROI from the image
    roi_cropped = image[y_start:y_end, x_start:x_end]

    # Add padding to the ROI
    padded_roi = cv2.copyMakeBorder(
        roi_cropped,
        top,
        bottom,
        left,
        right,
        cv2.BORDER_CONSTANT,
        value=color
    )

    return padded_roi


In [9]:
class YOLODetector:
    """
    YOLO detector, detections are upright bounding boxes with associated confidence.

    This class wraps a YOLO detection model and provides methods to detect objects in images.
    It uses the YOLOv11 model from the ultralytics package.
    """
    def __init__(self, yolo_model_path, obj_id=None):
        self.obj_id = obj_id # should be None for multi-class detector
        self.yolo = YOLO(yolo_model_path).cuda()
        self.yolo_confidence_thresh = 0.1
        self.image_size = 1280 # keep it as 1280, to be consistent with the YOLO model input size defined during training

    def detect(self, image, single_class_detector=False, rescale_factor=1.0):
        """
        Run YOLO on a single image.
        Returns a list of detections.
        Each detection is a dictionary with keys "bbox", "bb_center", "confidence".
        """
        results = self.yolo(image, imgsz=self.image_size, verbose=True)[0] # only result #0 as it's single image inference
        boxes = results.boxes.xyxy.cpu().numpy()
        confidences = results.boxes.conf.cpu().numpy()
        class_ids  = results.boxes.cls.cpu().numpy()
        class_names = results.names
        if len(results.boxes) == 0:
            return []
        
        valid = (confidences >= self.yolo_confidence_thresh)

        if single_class_detector:
            valid = valid & (class_ids == 0)  

        boxes = boxes[valid]
        confidences = confidences[valid]
        class_ids = class_ids[valid]
        detections = []

        should_rescale_output = np.abs(rescale_factor - 1.0) > 1e-6

        for box, confidence, class_id in zip(boxes, confidences, class_ids):
            x1, y1, x2, y2 = map(int, box)
            if should_rescale_output:
                x1 = int(x1 * rescale_factor)
                y1 = int(y1 * rescale_factor) 
                x2 = int(x2 * rescale_factor)
                y2 = int(y2 * rescale_factor)
        

            cx = 0.5 * (x1 + x2)
            cy = 0.5 * (y1 + y2)
            detections.append({
                'bbox': (x1, y1, x2, y2),
                'bb_center': (cx, cy),
                'confidence': confidence,
                'class_id': class_id,
                'class_name': class_names[class_id]
            })
        return detections


# TODO Fix YOLODetectorOrientedBoundingBox regarding subclassing from YOLODetector, and the coordinates of the bounding boxes mapping 
# to the original image, which is not correct.
if False:
    # This class is not used, but it is a good example of how to subclass the YOLODetector class to create a new detector.
    # It is not used because it is not working correctly, and it is not needed for the current task.
    # It is kept here for reference, in case we need to use it in the future.
    # The idea is to use this class to detect objects with oriented bounding boxes, using the YOLO detector.
    class YOLODetectorOrientedBoundingBox(YOLODetector):
        """
        YOLO detector with oriented bounding boxes.

        This class wraps a YOLO detection model that is configured to only detect upright bounding boxes, to handle objects that may appear at various orientations.
        It works by initially detecting objects on the original image and then, for each detection, it extracts a square crop centered
        around the object. The crop is then rotated at several angles, pasted into a blank image canvas, and the YOLO detector is applied on that canvas, once per each angle.
        Detections are filtered to retain only those that are well-centered in the rotated canvas to reject extraneous detections. 
        Finally, the most elongated bounding box is selected and translated back to the original image coordinates, and the highest confidence among all angles is kept.
        """
        def __init__(self, obj_id, yolo_model_path):
            self.obj_id = obj_id
            self.yolo = YOLO(yolo_model_path).cuda()
            self.yolo_conf_thresh = 0.1
            self.angle_step_deg = 360.0 / 16.0
            self.angle_values = np.linspace(0, 360, num=4, endpoint=True)[:-1].tolist() # Exclude 360 degrees last value, which equals 0 degrees
            # with num=4, we get 0, 90, 180, 270 degrees
            # with num=8, we get 0, 45, 90, 135, 180, 225, 270, 315 degrees
            # with num=16, we get 0, 22.5, 45, 67.5, 90, 112.5, 135, 157.5, 180, 202.5, 225, 247.5, 270, 292.5, 315, 337.5 degrees

        def _detect_single_image(self, image):
            """
            Run YOLO on a single image.
            Returns a list of detections.
            Each detection is a dictionary with keys "bbox", "bb_center", "confidence".
            """
            results = self.yolo(image, imgsz=1280, verbose=True)[0]
            boxes = results.boxes.xyxy.cpu().numpy()
            confs = results.boxes.conf.cpu().numpy()
            clss  = results.boxes.cls.cpu().numpy()
            if len(results.boxes) == 0:
                return []
            # Keep only detections with class==0 and conf>=threshold.
            valid = (clss == 0) & (confs >= self.yolo_conf_thresh)

            boxes = boxes[valid]
            confidences = confs[valid]
            detections = []
            for box, confidence in zip(boxes, confidences):
                x1, y1, x2, y2 = map(int, box)
                cx = 0.5 * (x1 + x2)
                cy = 0.5 * (y1 + y2)
                detections.append({
                    'bbox': (x1, y1, x2, y2),
                    'bb_center': (cx, cy),
                    'confidence': confidence
                })
            return detections

        def _detect_with_rotations(self, image):
            """
            Detect objects in several rotations of the image.
            This function applies the YOLO detector to each rotated version of image and 
            returns the detections with the highest confidences amongst all rotations.

            Each detection is a dictionary with keys "bbox", "angle", "bb_center", "confidence".

            The following steps are performed, in order:
            1. Detections are made on the original image. 
            2. For each detection, a square crop is made around the detection's center. The size of the crop is determined by the maximum of the width and height of the detection's bounding box.
            3. For each rotation angle:
                3.1 a grey image canvas is created,
                3.2 the cropped image is rotated by that angle and pasted onto the canvas, centered.
                3.3 the image is passed to the YOLO detector.
            4. The detections from all rotations are combined, and the one generating the most elongated bounding box is kept, associated with the highest confidence across all rotations.
            """

            # Start with the original image, no rotation:
            detections_zero_rotation = self._detect_single_image(image)


            # Resize image to 1280xR, where R is the aspect ratio of the original image:
            # Get the aspect ratio of the image:
            h, w = image.shape[:2]
            aspect_ratio = w / h
            # Resize the image to 1280xR:
            new_width = 1280
            new_height = int(new_width / aspect_ratio)
            image_resized = cv2.resize(image, (new_width, new_height), interpolation=cv2.INTER_LINEAR)

            if len(detections_zero_rotation) == 0:
                return []
            
            final_detections_list = []

            for detection_zero_rotation in detections_zero_rotation:
                detections = []
                detection_zero_rotation['angle'] = 0.0
                detections.append(detection_zero_rotation)

                SIZE_PADDING = 1.5

                x1, y1, x2, y2 = detection_zero_rotation['bbox']
                cx = 0.5 * (x1 + x2)
                cy = 0.5 * (y1 + y2)
                w = x2 - x1
                h = y2 - y1
                # Get the size of the square crop:
                size = int(max(w, h) * SIZE_PADDING)

                grey = (127, 127, 127)
                
                # Get a square crop around the detection.
                # For the parts where the crop is outside the image, we will use a grey color (127, 127, 127).
                # We don't change (cx,cy) because we want to keep the center of the crop in the same place.
                # Get the center of the crop in resized image coordinates:
                cx_crop = int(cx - 0.5 * size)
                cy_crop = int(cy - 0.5 * size)
                crop = extract_roi_with_padding(image, (cx_crop, cy_crop, size, size), color=grey)

                # Get the center of the crop in crop coordinates:
                cx_crop = int(size / 2)
                cy_crop = int(size / 2)

                # For each angle, rotate the crop and detect:

                for angle in self.angle_values:
                    # Create a grey image canvas:
                    canvas = grey[0] * np.ones((self.image_width, self.image_width, 3), dtype=np.uint8)
                    # Rotate the crop:
                    M = cv2.getRotationMatrix2D((cx_crop, cy_crop), angle, 1.0)
                    rotated_crop = cv2.warpAffine(crop, M, (size, size), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=grey)
                    # Paste the rotated crop onto the canvas, centered:
                    # Get the center of the canvas:
                    cx_canvas = int(self.image_width / 2)
                    cy_canvas = int(self.image_width / 2)
                    # Paste the rotated crop onto the canvas:
                    # Get the top-left corner of the crop in canvas coordinates:
                    x1_canvas = cx_canvas - cx_crop
                    y1_canvas = cy_canvas - cy_crop
                    canvas[y1_canvas:y1_canvas + size, x1_canvas:x1_canvas + size] = rotated_crop
                    # Detect on the rotated image:
                    detections_rotated = self.detect_single_image(canvas)
                    if len(detections_rotated) == 0:
                        continue
                    # Filter extraneous detections that are not centered in the canvas (coming from other objects in the scene):
                    # Add the angle to the detection:
                    for detection in detections_rotated:
                        bbcenter = detection['bb_center']
                        # distance between bbcenter and canvas center:
                        dist = np.linalg.norm(np.array(bbcenter) - np.array((cx_canvas, cy_canvas)))
                        # If the distance is too large, discard the detection:
                        if dist > 0.5 * (size / SIZE_PADDING): # (size / SIZE_PADDING) is the max of the width and height of the original detection's bounding box prior to cropping and rotation
                            continue
                        detection['angle'] = angle
                        detections.append(detection)

                # Get the most confident detection:
                if len(detections) == 0:
                    return []
                highest_confidence = max(d['confidence'] for d in detections)

                # Get the most elongated detection:
                best_detection = None
                best_aspect_ratio = 0
                for detection in detections:
                    x1, y1, x2, y2 = detection['bbox']
                    width = x2 - x1
                    height = y2 - y1
                    aspect_ratio = max(width / height, height / width)
                    if aspect_ratio > best_aspect_ratio:
                        best_aspect_ratio = aspect_ratio
                        best_detection = detection

                # Need to translate the detection back to the original image coordinates.
                # We won't use the angle to map back to the original image coordinates, because it is not needed.
                # We will use the angle later when we render the object.
                x1, y1, x2, y2 = best_detection['bbox']

                deltax = cx - cx_canvas
                deltay = cy - cy_canvas
                # Translate the detection back to the original image coordinates:
                best_detection['bbox'] = (x1 + deltax, y1 + deltay, x2 + deltax, y2 + deltay)
                # Translate the bb_center back to the original image coordinates:
                bb_center_x, bb_center_y = best_detection['bb_center']
                best_detection['bb_center'] = (bb_center_x + deltax, bb_center_y + deltay)

            
                final_detections_list.append(best_detection)

            return final_detections_list


In [10]:
# yolo_detectors_single_object:  the old approach: one model per object class.
# yolo_detector_multiclass: the new approach single model for all object classes approach with hillshade depth mapping

yolo_detectors_single_object = {}
object_meshes = {}


for this_obj_id in obj_ids:
    obj_id_path = str(1000000+this_obj_id)[1:]
    ply_file = os.path.join(models_dir, f"obj_{obj_id_path}.ply")
    
    yolo_model_path_phase1 = f'/home/joao/source/bpc-challenge/opencv/bpc/models_phase-1-backup/detection/obj_{this_obj_id}/yolo11-detection-obj_{this_obj_id}.pt'
    yolo_model_path_phase2 = f'/home/joao/source/bpc-challenge/opencv/bpc/models_phase-2-backup/detection/obj_{this_obj_id}/yolo11-detection-obj_{this_obj_id}.pt'
    
    
    if PHASE == 1:
        yolo_model_path = yolo_model_path_phase1
    elif PHASE == 2:
        yolo_model_path = yolo_model_path_phase2

    yolo_detectors_single_object[this_obj_id] = YOLODetector(yolo_model_path, this_obj_id)
    object_meshes[this_obj_id] = obj = trimesh.load(ply_file)

yolo_model_path_phase2_multiclass = os.path.join("/home/joao/source/bpc-challenge/opencv/bpc/models/detection", "yolo-11-training-08-single-model-grey-plus-depth-hillshade.pt")
yolo_detector_multiclass = YOLODetector(yolo_model_path_phase2_multiclass)

In [ ]:
TOTAL_IMAGES_TO_PROCESS = 10
for image_id in image_ids[:TOTAL_IMAGES_TO_PROCESS]:
    dummy_obj_id = 99999

    # Get only camera 1 (index 0):
    capture = Capture.from_dir(scene_dir, cam_ids, image_id, dummy_obj_id)
    image_cam_1 = capture.images[0].copy()
    #image_cam_1 = du.apply_image_transformations(image_cam_1)
    capture.images[0] = image_cam_1
    image_cam_1 = capture.images[0].copy()
    depth_cam_1 = capture.depths[0].copy()
    intrinsics_K_cam_1 = capture.Ks[0].copy()

    depth_cam_1_tranformed_metric_scale = du.transform_depth_image(depth_cam_1, DEPTH_IMAGE_SCALE_PX2MM, max_depth_mm=5000.0)

    # Infer for all object IDs at once, then apply inter-class filtering:
    detections_all_obj_ids = {}

    USE_OLD_YOLO_SINGLE_CLASS_APPROACH = False

    if USE_OLD_YOLO_SINGLE_CLASS_APPROACH:
        for this_obj_id in obj_ids:
            detections_this_obj_id = yolo_detectors_single_object[this_obj_id].detect(image_cam_1)
            detections_all_obj_ids[this_obj_id] = detections_this_obj_id

    else: # use the new approach with single YOLO model for all object classes with hillshade depth mapping
        yolo_input = du.compose_grey_plus_hillshade_depth_image(image_cam_1, depth_cam_1, 
                                                                new_width=yolo_detector_multiclass.image_size, 
                                                                is_synthetic=True)

        # Plot each of the RGB channels of yolo_input:
        plt.figure(figsize=(15,15))
        plt.subplot(131)
        plt.imshow(yolo_input[:,:,0], cmap='gray')
        plt.title('Grayscale Channel')
        plt.subplot(132) 
        plt.imshow(yolo_input[:,:,1], cmap='gray')
        plt.title('Hillshade 0° Azimuth')
        plt.subplot(133)
        plt.imshow(yolo_input[:,:,2], cmap='gray')
        plt.title('Hillshade 135° Azimuth')
        plt.show()

        # I accidentally inverted the order of channels when running the data preparation pipeline (prepare_data.py)
        # and now my YOLO model must be fed with the channels reversed as well!     (-__-)'
        yolo_input = cv2.cvtColor(yolo_input, cv2.COLOR_RGB2BGR)

        max_original_image_side = np.max(image_cam_1.shape)
        rescale_factor = max_original_image_side / yolo_detector_multiclass.image_size
        all_detections = yolo_detector_multiclass.detect(yolo_input, rescale_factor=rescale_factor)
        
        for detection in all_detections:
            obj_id = int(detection['class_id']) # they map by index value 1-to-1 on phase 2
            if obj_id not in detections_all_obj_ids:
                detections_all_obj_ids[obj_id] = []
            detections_all_obj_ids[obj_id].append(detection)


    do_filter_detections = True
    if do_filter_detections:
        # Filter out detections with confidence < 0.4:
        for this_obj_id, detections_this_id in detections_all_obj_ids.items():
            detections_all_obj_ids[this_obj_id] = [detection for detection in detections_this_id if detection['confidence'] >= 0.4]

        detections_all_obj_ids = ydf.select_most_confident_detections(detections_all_obj_ids)

        # I ended up disabling this one because I saw some true positives being rejected. Imagine you have multiple rods laid out
        # close and parallel to each other... Training YOLO to produce oriented bounding boxes would have helped immensely here!
        #detections_all_obj_ids = ydf.select_most_confident_detections(detections_all_obj_ids, object_ids_group=[4, 8], iou_threshold=0.2)

        # Check lower bound for objects that are very unlikely to be put standing upright on a flat surface due to their geometry
        # For all others, we cannot check lower bound, as the aspect ratio of the object apperance on the image can change dramatically.
        lower_bound_active_for_obj_ids =[]
        #lower_bound_active_for_obj_ids = [0, 6, 8] I am going to disable this. It just occured to me that these objects can be put slanted on a bin
        # against one of the bin's walls.
        detections_all_obj_ids = ydf.filter_detections_by_appearance_and_metric_size(
            detections_all_obj_ids, 
            depth_cam_1_tranformed_metric_scale,
            intrinsics_K_cam_1, 
            object_meshes, 
            lower_bound_active_for_obj_ids)
        
        # It is better if this inter-class enclosure check is made only after metric size checks, because of directional lighting
        # casting a long shadow from a tall upright object, where the shadow is deemed a false positive, potentially yielding
        # the target object as a false negative, given the enclosure check.
        detections_all_obj_ids = ydf.filter_enclosed_detections_across_classes(detections_all_obj_ids, excluded_elongated_object_ids=[4, 8, 9])


    # Now we can draw the filtered detections on the image:
    for this_obj_id, detections_this_id in detections_all_obj_ids.items():
        # For each detection, draw the object ID, bounding box, confidence:
        for detection_this_id in detections_this_id:
            bbox = detection_this_id['bbox']
            confidence = detection_this_id['confidence']
            bb_center = detection_this_id['bb_center']

            text = f"ID: {this_obj_id} Conf: {confidence:.2f}"
            font = cv2.FONT_HERSHEY_SIMPLEX
            font_scale = 1.4  # Increased by 100%
            thickness = 2
            (text_width, text_height) = cv2.getTextSize(text, font, font_scale, thickness)[0]
            
            # Set the background color for the text
            bg_color = du.get_color_for_class_id(this_obj_id)
            text_color = (255, 255, 255)  # White color for the text

            # Calculate the rectangle coordinates for the background
            rect_x = int(bbox[0])
            rect_y = int(bbox[1] - 50)  # Position the rectangle above the bounding box
            rect_w = text_width + 10
            rect_h = text_height + 20

            # Draw the background rectangle
            cv2.rectangle(image_cam_1, (rect_x, rect_y), (rect_x + rect_w, rect_y + rect_h), bg_color, -1)

            # Draw the object bounding box
            cv2.rectangle(image_cam_1, (int(bbox[0]), int(bbox[1])), (int(bbox[2]), int(bbox[3])), du.get_color_for_class_id(this_obj_id), 4)

            # Put the text on the image
            cv2.putText(image_cam_1, text, (int(bbox[0]) + 5, int(bbox[1] - 10)), font, font_scale, text_color, thickness)

    # Show the image with detections
    plt.figure(figsize=(15, 15))
    plt.imshow(image_cam_1)
    plt.axis('off')
    plt.show()


In [12]:
if False:
    # scene_dir = "./datasets/ipd_bop_data_jan25_1/train_pbr/000000/"
    # models_dir = './datasets/ipd_bop_data_jan25_1/models_eval/'
    scene_dir = "./datasets/ipd/test/000003/"
    models_dir = './datasets/ipd/models_eval/'
    cam_ids = ["cam1", "cam2", "cam3"]
    image_id = 2
    this_obj_id = 11
    obj_id_path = str(1000000+this_obj_id)[1:]
    ply_file = os.path.join(models_dir, f"obj_{obj_id_path}.ply")
    obj = trimesh.load(ply_file)
    yolo_model_path = f'bpc/yolo/models/detection/obj_{this_obj_id}/yolo11-detection-obj_{this_obj_id}.pt'
    pose_model_path = f'bpc/pose/pose_checkpoints/obj_{this_obj_id}/final_model.pth'

    pose_params = PoseEstimatorParams(
        yolo_model_path=yolo_model_path,
        pose_model_path=pose_model_path, 
        yolo_conf_thresh=0.01,
    )
    pose_estimator = PoseEstimator(pose_params)
    t = time.time()
    capture = Capture.from_dir(scene_dir, cam_ids, image_id, this_obj_id)
    detections_this_id = pose_estimator._detect(capture)
    pose_predictions = pose_estimator._match(capture, detections_this_id)
    pose_estimator._estimate_rotation(pose_predictions)
    print(time.time() - t)

    for idx in range(len(capture.Ks)):
        plt.figure(figsize=(15, 15))
        plt.imshow(capture.images[idx])
        a, b = render_mask(obj, capture.Ks[idx], (capture.RTs[idx]), capture.images[0].shape[:2][::-1], [x.pose for x in pose_predictions])
        plt.imshow(a, alpha=0.5)
        plt.show()
